# Debugger and Profiler

In [16]:
# install dependencies
!pip install smdebug

In [17]:
hyperparameters = {
    "batch_size": 2048,
    "gpu": True,
    "epoch": 2,
    "model": "resnet50",
}

In [18]:
from sagemaker.core.debugger import Rule, ProfilerRule, rule_configs

rules = [ 
    Rule.sagemaker(rule_configs.vanishing_gradient()),
    Rule.sagemaker(rule_configs.overfit()),
    Rule.sagemaker(rule_configs.overtraining()),
    Rule.sagemaker(rule_configs.poor_weight_initialization()),
]
#TODO: Can you add the rules you want to track

In [19]:
from sagemaker.core.debugger import DebuggerHookConfig, ProfilerConfig, FrameworkProfile

#TODO: Can you create the profiler and debugger configs
profiler_config = ProfilerConfig(system_monitor_interval_millis=500, framework_profile_params=FrameworkProfile(num_steps=10))
debugger_config = DebuggerHookConfig(hook_parameters={"train.save_interval": "100", "eval.save_interval": "10"})

In [20]:
# Added block 
import os

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sagemaker_session = Session()
role = get_execution_role()
region = boto3.session.Session().region_name
bucket = sagemaker_session.default_bucket()

In [21]:
import sagemaker
from sagemaker.pytorch import PyTorch

estimator = estimator = PyTorch(
    role=role,
    instance_count=1,
    instance_type="ml.g4dn.2xlarge",
    entry_point="scripts/pytorch_cifar_profiling.py",
    framework_version="1.8",
    py_version="py36",
    hyperparameters=hyperparameters,
    profiler_config=profiler_config,
    debugger_hook_config=debugger_config,
    rules=rules,
)

#TODO: Create the estimator to train your model

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│    1 import sagemaker                                                                            │
│ ❱  2 from sagemaker.pytorch import PyTorch                                                       │
│    3                                                                                             │
│    4 estimator = estimator = PyTorch(                                                            │
│    5 │   role=role,                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/deprecations.py:319 in find_spec          │
│                                                                                                  │
│   316 │   │   if leaf in _V3_REPLACEMENTS:                                                       │
│   317 │   │   │   # Curated, high-traffic module -> precise guidance.                            │
│   318 │   │   │   replacement, v3_import, docs_module = _V3_REPLACEMENTS[leaf]                   │
│ ❱ 319 │   │   │   raise_removed_in_v3(                                                           │
│   320 │   │   │   │   module=fullname,                                                           │
│   321 │   │   │   │   replacement=replacement,                                                   │
│   322 │   │   │   │   v3_import=v3_import,                                                       │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/deprecations.py:280 in                    │
│ raise_removed_in_v3                                                                              │
│                                                                                                  │
│   277 │   # leave a breadcrumb for log-captured environments without duplicating the             │
│   278 │   # message at WARNING level.                                                            │
│   279 │   logger.debug(msg)                                                                      │
│ ❱ 280 │   raise ModuleNotFoundError(msg, name=module)                                            │
│   281                                                                                            │
│   282                                                                                            │
│   283 class _RemovedV2ModuleFinder(importlib.abc.MetaPathFinder):                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ModuleNotFoundError: `sagemaker.pytorch` was removed in the SageMaker Python SDK v3. Use `ModelTrainer`. (from 
sagemaker.train import ModelTrainer)
Docs: https://sagemaker.readthedocs.io/en/stable/api/generated/sagemaker.train.model_trainer.html
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide.

In [ ]:
estimator.fit(wait=True)

In [ ]:
import boto3

session = boto3.session.Session()
region = session.region_name

training_job_name = estimator.latest_training_job.name
print(f"Training jobname: {training_job_name}")
print(f"Region: {region}")

In [ ]:
from smdebug.trials import create_trial
from smdebug.core.modes import ModeKeys

trial = create_trial(estimator.latest_job_debugger_artifacts_path())

In [ ]:
# TODO: Can you print the names of all the tensors that were tracked
# TODO: Can you print the number of datapoints for one of those tensors
# for both train and eval mode

In [ ]:
from smdebug.profiler.analysis.notebook_utils.training_job import TrainingJob

tj = TrainingJob(training_job_name, region)
tj.wait_for_sys_profiling_data_to_be_available()

In [ ]:
from smdebug.profiler.analysis.notebook_utils.timeline_charts import TimelineCharts

system_metrics_reader = tj.get_systems_metrics_reader()
system_metrics_reader.refresh_event_file_list()

view_timeline_charts = TimelineCharts(
    system_metrics_reader,
    framework_metrics_reader=None,
    select_dimensions=["CPU", "GPU"],
    select_events=["total"],
)

In [ ]:
rule_output_path = estimator.output_path + estimator.latest_training_job.job_name + "/rule-output"
print(f"You will find the profiler report in {rule_output_path}")

In [ ]:
! aws s3 ls {rule_output_path} --recursive

In [ ]:
! aws s3 cp {rule_output_path} ./ --recursive

In [ ]:
import os

# get the autogenerated folder name of profiler report
profiler_report_name = [
    rule["RuleConfigurationName"]
    for rule in estimator.latest_training_job.rule_job_summary()
    if "Profiler" in rule["RuleConfigurationName"]
][0]

In [ ]:
import IPython

IPython.display.HTML(filename=profiler_report_name + "/profiler-output/profiler-report.html")